# 🟢 Notebook 01 — Connexion PostgreSQL avec psycopg2
**Projet fil rouge MobiGreen Urban | Module SQL — Kit 2, Itération 1**

---

## Objectifs
- Établir une connexion à PostgreSQL avec **psycopg2**
- Exécuter des requêtes SQL et récupérer les résultats en Python
- Utiliser les **requêtes paramétrées**
- Explorer les tables de votre base MobiGreen

## Pré-requis
- Base `mobigreen_urban` opérationnelle (Docker du Kit 1)
- `pip install psycopg2-binary` (déjà fait si vous avez cloné le projet GitHub)

---

## 1. Configuration de la connexion

**À faire :** Importez `psycopg2`, `os` et `dotenv`. Chargez les variables d'environnement depuis le fichier `.env` situé à la racine du projet, puis construisez le dictionnaire `DB_CONFIG` à partir de ces variables.

> ⚠️ **Bonne pratique :** Ne codez **jamais** vos identifiants en dur dans le code. Utilisez toujours des variables d'environnement chargées depuis un fichier `.env` (qui est ignoré par Git).

```python
# Exemple de structure attendue :
from dotenv import load_dotenv
import os

load_dotenv("../.env")  # charge les variables depuis le fichier .env

DB_CONFIG = {
    "dbname":   os.getenv("DB_NAME"),
    "user":     os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "host":     os.getenv("DB_HOST"),
    "port":     os.getenv("DB_PORT"),
}
```

Testez la connexion en ouvrant puis refermant une connexion, et affichez un message de confirmation.

In [6]:
from dotenv import load_dotenv
import os
import psycopg2   
load_dotenv("../.env")  # charge les variables depuis le fichier .env

DB_CONFIG = {
    "dbname":   os.getenv("DB_NAME"),
    "user":     os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "host":     os.getenv("DB_HOST"),
    "port":     os.getenv("DB_PORT"),
    
}
try:
    connection = psycopg2.connect(**DB_CONFIG)
    print("Connexion à la base de données réussie !")
finally:
    # Fermeture propre de la connexion si elle a été ouverte
    if 'connection' in locals() and connection:
        connection.close()
        print("Connexion fermée.")

Connexion à la base de données réussie !
Connexion fermée.


## 2. Fonction utilitaire

**À faire :** Écrivez un **context manager** `get_cursor()` qui ouvre une connexion et un curseur, gère le commit/rollback, et ferme proprement dans tous les cas.

Écrivez également une fonction `exec_query(sql, params=None)` qui exécute une requête SELECT et retourne les noms de colonnes et les lignes de résultats.

> 💡 Indice : utilisez `cursor.description` pour récupérer les noms de colonnes.

In [7]:

from contextlib import contextmanager

@contextmanager
def get_cursor():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    try:
        yield cur
        conn.commit()
    except Exception as e:
        #conn.rollback()
        print(f" Rollback — erreur : {e}")
        raise e
    finally:
        cur.close()
        conn.close()

def print_results(columns, rows, max_rows=20):
    """Affiche les résultats d'une requête sous forme de tableau lisible."""
    if not rows:
        print("  (aucun résultat)")
        return

    # Calcul de la largeur de chaque colonne
    col_widths = [len(str(c)) for c in columns]
    for row in rows[:max_rows]:
        for i, val in enumerate(row):
            col_widths[i] = max(col_widths[i], len(str(val)))

    sep  = "+" + "+".join("-" * (w + 2) for w in col_widths) + "+"
    head = "|" + "|".join(f" {c:<{w}} " for c, w in zip(columns, col_widths)) + "|"

    print(sep)
    print(head)
    print(sep)
    for row in rows[:max_rows]:
        print("|" + "|".join(f" {str(v):<{w}} " for v, w in zip(row, col_widths)) + "|")
    print(sep)

    if len(rows) > max_rows:
        print(f"  … {len(rows) - max_rows} lignes supplémentaires masquées (total : {len(rows)})")
    print(f"  → {len(rows)} ligne(s) retournée(s)\n")

In [8]:

def exec_query(sql, params=None):
    with get_cursor() as cur:
        cur.execute(sql, params)
        # Récupère les noms de colonnes depuis cursor.description
        columns = [desc[0] for desc in cur.description]
        rows = cur.fetchall()
    return columns, rows

columns, rows = exec_query("SELECT * FROM donnees_airs")
print("Colonnes :", columns)
for row in rows:
    print(row)
    
  

Colonnes : ['donnees_air_id', 'station_id', 'indice_qualite', 'pm25', 'pm10', 'no2', 'horodatage', 'created_at']
(1, 1, Decimal('42.50'), Decimal('12.30'), Decimal('20.10'), Decimal('15.00'), datetime.datetime(2026, 4, 1, 8, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 2, 11, 38, 50, 269365, tzinfo=datetime.timezone.utc))
(2, 2, Decimal('65.80'), Decimal('25.40'), Decimal('40.20'), Decimal('30.10'), datetime.datetime(2026, 4, 1, 9, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 2, 11, 38, 50, 269365, tzinfo=datetime.timezone.utc))
(3, 3, Decimal('120.00'), Decimal('55.00'), Decimal('80.00'), Decimal('60.50'), datetime.datetime(2026, 4, 1, 10, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 2, 11, 38, 50, 269365, tzinfo=datetime.timezone.utc))
(4, 4, Decimal('85.30'), Decimal('30.20'), Decimal('50.10'), Decimal('40.00'), datetime.datetime(2026, 4, 1, 11, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 2, 11, 38, 50, 269365, tzinfo=

## 3. Exploration du schéma

**À faire :** Interrogez `information_schema.tables` pour lister toutes les tables du schéma `public` de votre base.

Ensuite, comptez le nombre de lignes dans chacune de vos tables en exécutant un `COUNT(*)` pour chacune.

In [25]:
sql_tables = """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
      AND table_type   = 'BASE TABLE'
    ORDER BY table_name;
"""
cols, rows = exec_query(sql_tables)
tables = [row[0] for row in rows]

print(f"Tables trouvées dans le schéma public ({len(tables)}) :")
for t in tables:
    print(f"  - {t}")


print(f"{'Table':<30} {'Nb lignes':>10}")
print("-" * 55)

for table in tables:
    sql_count = f"SELECT COUNT(*) FROM {table};"
    _, result = exec_query(sql_count)
    count = result[0][0]
    barre = "█" * min(count, 40)  # barre visuelle proportionnelle
    print(f"  {table:<28} {count:>6}   {barre}")

Tables trouvées dans le schéma public (14) :
  - abonnements
  - analyse_secteurs
  - communes
  - donnees_airs
  - donnees_meteos
  - incidents
  - localisation_station
  - paiements
  - point_trajets
  - stations
  - trajets
  - utilisateurs
  - vehicules
  - zones_metro
Table                           Nb lignes
-------------------------------------------------------
  abonnements                       0   
  analyse_secteurs                  0   
  communes                          0   
  donnees_airs                      5   █████
  donnees_meteos                   10   ██████████
  incidents                         5   █████
  localisation_station              7   ███████
  paiements                         0   
  point_trajets                     0   
  stations                          7   ███████
  trajets                          20   ████████████████████
  utilisateurs                     10   ██████████
  vehicules                        15   ███████████████
  zones_metro   

## 4. Requêtes SELECT simples

**À faire :** Écrivez et exécutez une requête SELECT sur chacune de vos tables principales. Affichez les résultats de façon lisible.

Pour chaque table, vérifiez que les données insérées en Kit 1 sont bien présentes.

In [4]:
print("=== Utilisateurs ===")
cols, rows = exec_query("""
    SELECT nom, email, date_inscription, actif, solde
    FROM utilisateurs
    ORDER BY date_inscription;
""")
print_results(cols, rows)

print("=== Véhicules ===")
cols, rows = exec_query("""
    SELECT immatriculation, type, statut, autonomie_km, derniere_maintenance
    FROM vehicules
    ORDER BY type, statut;
""")
print_results(cols, rows)

print("=== Abonnements ===")
cols, rows = exec_query("""
    SELECT abonnement_id, utilisateur_id, type, date_debut, date_fin, prix, actif
    FROM abonnements
    ORDER BY type;
""")
print_results(cols, rows)

=== Utilisateurs ===
+----------------+----------------------------+----------------------------+-------+--------+
| nom            | email                      | date_inscription           | actif | solde  |
+----------------+----------------------------+----------------------------+-------+--------+
| Jean Dupont    | jean.dupont@example.com    | 2026-04-02 11:33:09.401614 | True  | 150.50 |
| Marie Martin   | marie.martin@example.com   | 2026-04-02 11:33:09.401614 | True  | 200.00 |
| Paul Durand    | paul.durand@example.com    | 2026-04-02 11:33:09.401614 | True  | 0.00   |
| Sophie Bernard | sophie.bernard@example.com | 2026-04-02 11:33:09.401614 | True  | 75.20  |
| Lucas Petit    | lucas.petit@example.com    | 2026-04-02 11:33:09.401614 | False | 10.00  |
| Emma Robert    | emma.robert@example.com    | 2026-04-02 11:33:09.401614 | True  | 320.99 |
| Hugo Richard   | hugo.richard@example.com   | 2026-04-02 11:33:09.401614 | True  | 5.75   |
| Chloé Moreau   | chloe.moreau@example

## 5. Requêtes avec jointures

**À faire :** Écrivez au moins deux requêtes impliquant des jointures entre plusieurs tables de votre schéma. Choisissez des jointures qui ont du sens métier pour MobiGreen.

> Exemples de questions auxquelles répondre (adaptez à votre schéma) :
> - Quels trajets ont été effectués, avec le nom de l'utilisateur et le type de véhicule ?
> - Quelles stations sont rattachées à quelle zone géographique ?
> - Quels véhicules se trouvent actuellement dans quelle station ?

In [12]:
cols, rows = exec_query("""
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'zones_metro';
""")

print(rows)

[('zone_id',), ('nom',), ('code_insee',), ('created_at',)]


In [13]:
# Trajets avec nom de l'utilisateur et type de véhicule
print("=== Trajets : utilisateur + véhicule ===")
cols, rows = exec_query("""
    SELECT
        u.nom  AS utilisateur,
        v.immatriculation AS vehicule,
        v.type AS type_vehicule,
        t.date_debut,
        t.duree_minutes,
        t.distance_km,
        t.montant_facture,
        t.statut
    FROM trajets t
    JOIN utilisateurs u ON u.utilisateur_id = t.utilisateur_id
    JOIN vehicules    v ON v.vehicule_id = t.vehicule_id
    ORDER BY t.date_debut DESC;
""")
print_results(cols, rows)


# Stations avec leur zone géographique et coordonnées GPS
print("=== Stations : zones + coordonnées GPS ===")
cols, rows = exec_query("""
    SELECT
        s.nom AS station,
        s.adresse,
        z.nom AS zone,
        ls.latitude,
        ls.longitude,
        s.capacite_totale,
        s.places_disponibles
    FROM stations s
    JOIN zones_metro z ON z.zone_id = s.zone_id
    JOIN localisation_station ls ON ls.station_id = s.station_id
    ORDER BY z.nom, s.nom;
""")
print_results(cols, rows)

=== Trajets : utilisateur + véhicule ===
+----------------+------------+-----------------+---------------------------+---------------+-------------+-----------------+----------+
| utilisateur    | vehicule   | type_vehicule   | date_debut                | duree_minutes | distance_km | montant_facture | statut   |
+----------------+------------+-----------------+---------------------------+---------------+-------------+-----------------+----------+
| Léa Laurent    | FR-VEH-007 | vélo électrique | 2026-03-26 17:00:00+00:00 | 35            | 11.000      | 5.50            | terminé  |
| Nathan Simon   | FR-VEH-006 | trottinette     | 2026-03-26 16:00:00+00:00 | 25            | 7.000       | 3.50            | terminé  |
| Chloé Moreau   | FR-VEH-005 | vélo            | 2026-03-26 15:00:00+00:00 | 5             | 1.000       | 0.50            | terminé  |
| Hugo Richard   | FR-VEH-004 | voiture         | 2026-03-26 14:00:00+00:00 | 20            | 10.000      | 6.00            | terminé  |


In [14]:
with get_cursor() as cur:
    cur.execute("""
        INSERT INTO localisation_station (station_id, latitude, longitude)
        VALUES
        (1, 45.1860, 5.7244),
        (2, 45.1940, 5.7068),
        (3, 45.1799, 5.7293),
        (4, 45.1933, 5.7672),
        (5, 45.1488, 5.7229),
        (6, 45.2095, 5.7731),
        (7, 45.1920, 5.6890);
    """)

In [15]:
# Stations avec leur zone géographique et coordonnées GPS
print("=== Stations : zones + coordonnées GPS ===")
cols, rows = exec_query("""
    SELECT
        s.nom AS station,
        s.adresse,
        z.nom AS zone,
        ls.latitude,
        ls.longitude,
        s.capacite_totale,
        s.places_disponibles
    FROM stations s
    JOIN zones_metro z ON z.zone_id = s.zone_id
    JOIN localisation_station ls ON ls.station_id = s.station_id
    ORDER BY z.nom, s.nom;
""")
print_results(cols, rows)

=== Stations : zones + coordonnées GPS ===
+----------------------+----------------------------------------+----------------------+------------+-----------+-----------------+--------------------+
| station              | adresse                                | zone                 | latitude   | longitude | capacite_totale | places_disponibles |
+----------------------+----------------------------------------+----------------------+------------+-----------+-----------------+--------------------+
| Échirolles Centre    | 1 pl. de la République, Échirolles     | Echirolles           | 45.1488000 | 5.7229000 | 10              | 3                  |
| Fontaine - Piscine   | 8 rue des Sports, Fontaine             | Fontaine             | 45.1920000 | 5.6890000 | 8               | 4                  |
| Hôtel de Ville       | 11 bd. Jean Pain, Grenoble             | Grenoble             | 45.1860000 | 5.7244000 | 12              | 5                  |
| Paul Mistral - Parc  | 1 av. Paul Mis

## 6. Requêtes paramétrées

**À faire :** Réécrivez au moins une de vos requêtes précédentes sous forme de **fonction Python** qui accepte un paramètre de filtrage.

> ⚠️ Rappel de sécurité : utilisez toujours la syntaxe `%s` de psycopg2 pour les paramètres. Ne construisez jamais une requête par concaténation de chaînes.

Testez votre fonction avec plusieurs valeurs différentes.

In [17]:
# Véhicules disponibles avec leur station actuelle
print("=== Véhicules disponibles par station ===")
cols, rows = exec_query("""
    SELECT
        s.nom AS station,
        v.immatriculation,
        v.type,
        v.statut,
        v.autonomie_km
    FROM vehicules v
    JOIN stations  sON s.station_id = v.station_id
    WHERE v.statut = 'disponible'
    ORDER BY s.nom, v.type;
""")
print_results(cols, rows)

=== Véhicules disponibles par station ===
+----------------------+-----------------+-----------------+------------+--------------+
| station              | immatriculation | type            | statut     | autonomie_km |
+----------------------+-----------------+-----------------+------------+--------------+
| Campus universitaire | FR-VEH-006      | trottinette     | disponible | 30.00        |
| Hôtel de Ville       | FR-VEH-014      | trottinette     | disponible | 28.75        |
| Hôtel de Ville       | FR-VEH-001      | vélo            | disponible | 0.00         |
| Hôtel de Ville       | FR-VEH-009      | vélo            | disponible | 0.00         |
| Hôtel de Ville       | FR-VEH-003      | vélo électrique | disponible | 60.00        |
| Échirolles Centre    | FR-VEH-008      | voiture         | disponible | 500.00       |
| Échirolles Centre    | FR-VEH-013      | vélo            | disponible | 0.00         |
+----------------------+-----------------+-----------------+--------

In [19]:
# 5.4 — Incidents avec véhicule concerné et utilisateur déclarant
print("=== Incidents : véhicule + déclarant ===")
cols, rows = exec_query("""
    SELECT
        i.titre,
        i.statut                AS statut_incident,
        u.nom                   AS signale_par,
        v.immatriculation       AS vehicule,
        v.type                  AS type_vehicule,
        i.date_signalement,
        i.date_resolution
    FROM incidents i
    JOIN utilisateurs u ON u.utilisateur_id = i.utilisateur_id
    LEFT JOIN vehicules v ON v.vehicule_id = i.vehicule_id
    ORDER BY i.date_signalement DESC;
""")
print_results(cols, rows)

=== Incidents : véhicule + déclarant ===
+-------------------+-----------------+----------------+------------+-----------------+---------------------------+---------------------------+
| titre             | statut_incident | signale_par    | vehicule   | type_vehicule   | date_signalement          | date_resolution           |
+-------------------+-----------------+----------------+------------+-----------------+---------------------------+---------------------------+
| Chaîne cassée     | en_cours        | Lucas Petit    | FR-VEH-001 | vélo            | 2026-03-31 16:10:00+00:00 | None                      |
| Panne moteur      | ouvert          | Paul Durand    | FR-VEH-004 | voiture         | 2026-03-30 10:00:00+00:00 | None                      |
| Batterie faible   | en_cours        | Marie Martin   | FR-VEH-003 | vélo électrique | 2026-03-29 09:15:00+00:00 | None                      |
| Problème de frein | résolu          | Jean Dupont    | FR-VEH-002 | trottinette     | 2026-03

## 7. Agrégations

**À faire :** Écrivez des requêtes utilisant des fonctions d'agrégation (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`) avec `GROUP BY` sur vos données MobiGreen.

> Exemples :
> - Nombre de trajets par utilisateur
> - Distance ou durée moyenne selon le type de véhicule
> - Nombre de véhicules par statut

In [24]:
# Trajets d'un utilisateur (filtré par email)


def get_trajets_utilisateur(email):
 
    sql = """
        SELECT
            u.nom AS utilisateur,
            v.type AS vehicule,
            t.date_debut,
            t.duree_minutes,
            t.distance_km,
            t.montant_facture,
            t.statut
        FROM trajets t
        JOIN utilisateurs u ON u.utilisateur_id = t.utilisateur_id
        JOIN vehicules    v ON v.vehicule_id = t.vehicule_id
        WHERE u.email = %s
          AND t.statut = 'terminé'
        ORDER BY t.date_debut DESC;
    """
    cols, rows = exec_query(sql, (email,))
    print(f"Trajets de '{email}' :")
    print_results(cols, rows)


# Test avec plusieurs valeurs
get_trajets_utilisateur("jean.dupont@example.com")
get_trajets_utilisateur("nathan.simon@example.com")
get_trajets_utilisateur("lea.laurent@example.com")

Trajets de 'jean.dupont@example.com' :
+-------------+-------------+---------------------------+---------------+-------------+-----------------+---------+
| utilisateur | vehicule    | date_debut                | duree_minutes | distance_km | montant_facture | statut  |
+-------------+-------------+---------------------------+---------------+-------------+-----------------+---------+
| Jean Dupont | voiture     | 2026-03-26 08:00:00+00:00 | 40            | 22.000      | 13.20           | terminé |
| Jean Dupont | trottinette | 2026-03-25 08:00:00+00:00 | 25            | 6.500       | 3.25            | terminé |
+-------------+-------------+---------------------------+---------------+-------------+-----------------+---------+
  → 2 ligne(s) retournée(s)

Trajets de 'nathan.simon@example.com' :
+--------------+-------------+---------------------------+---------------+-------------+-----------------+---------+
| utilisateur  | vehicule    | date_debut                | duree_minutes | dist

---
## Récapitulatif — Ce que vous avez mis en œuvre

Complétez ce tableau avant de passer au notebook suivant :

| Concept | Syntaxe utilisée | ✅ Testé |
|---|---|---|
| Connexion | `psycopg2.connect(...)` | ☐ |
| Curseur | `conn.cursor()` | ☐ |
| Exécution | `cursor.execute(sql, params)` | ☐ |
| Récupération | `fetchall()` / `fetchone()` | ☐ |
| Paramètre sécurisé | `%s` avec tuple | ☐ |
| Context manager | `with get_cursor() as c:` | ☐ |
| Agrégation | `COUNT / GROUP BY` | ☐ |

---
**Suite :** `notebook_02_pandas.ipynb`